In [1]:
# Generate Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Reading from file")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
# load the dataset
df_raw = spark.read.format ("text").load("/home/jovyan/data/dataset.txt")

In [3]:
df_raw.printSchema()

root
 |-- value: string (nullable = true)



In [4]:
df_raw.show()

+--------------------+
|               value|
+--------------------+
|This is our pyspa...|
+--------------------+



In [5]:
# Split the line into words
from pyspark.sql.functions import split

df_words = df_raw.withColumn("words", split("value", " "))

In [6]:
df_words.show()

+--------------------+--------------------+
|               value|               words|
+--------------------+--------------------+
|This is our pyspa...|[This, is, our, p...|
+--------------------+--------------------+



In [32]:

# Explode the list of words
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words"))

In [14]:
df_explode.show()

+--------------------+--------------------+-----+
|               value|               words| word|
+--------------------+--------------------+-----+
|simon had a dog a...|[simon, had, a, d...|simon|
|simon had a dog a...|[simon, had, a, d...|  had|
|simon had a dog a...|[simon, had, a, d...|    a|
|simon had a dog a...|[simon, had, a, d...|  dog|
|simon had a dog a...|[simon, had, a, d...|  and|
|simon had a dog a...|[simon, had, a, d...|    a|
|simon had a dog a...|[simon, had, a, d...|  cat|
|simon had a dog a...|[simon, had, a, d...|  the|
|simon had a dog a...|[simon, had, a, d...|  dog|
|simon had a dog a...|[simon, had, a, d...|  and|
|simon had a dog a...|[simon, had, a, d...|  cat|
|simon had a dog a...|[simon, had, a, d...| used|
|simon had a dog a...|[simon, had, a, d...|   to|
|simon had a dog a...|[simon, had, a, d...| love|
|simon had a dog a...|[simon, had, a, d...|simon|
|simon had a dog a...|[simon, had, a, d...|     |
+--------------------+--------------------+-----+


In [33]:
# Explode the list of words
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words")).drop("value", "words")

In [16]:
df_explode.show()

+-----+
| word|
+-----+
|simon|
|  had|
|    a|
|  dog|
|  and|
|    a|
|  cat|
|  the|
|  dog|
|  and|
|  cat|
| used|
|   to|
| love|
|simon|
|     |
+-----+



In [34]:
# Aggregate the words to generate count
from pyspark.sql.functions import count, lit

df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("cnt"))

In [18]:
df_agg.show()

+-----+---+
| word|cnt|
+-----+---+
| used|  1|
|simon|  2|
|  dog|  2|
| love|  1|
|  had|  1|
|  cat|  2|
|  the|  1|
|  and|  2|
|    a|  2|
|     |  1|
|   to|  1|
+-----+---+



In [ ]:
# Write the output to console streaming

df_agg.writeStream.format("console").outputMode("complete").start().awaitTermination()

In [ ]:
# Read input data

df_raw = spark.readStream.format("socket").option("host","localhost").option("port", "9999").load()